# Conditioning assertions
Standalone tests for `build_params` (key/shape merge asserts) and the identity-init no-op.

In [8]:
# Put the repo root on the path and make it the CWD (so `src.*` imports and
# relative paths like configs/config.yaml resolve). Absolute path = idempotent.
import sys, os
REPO_ROOT = "/home/rhautier/ddpm-jax"
sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
print(os.getcwd(), "| config found:", os.path.exists("configs/config.yaml"))

/home/rhautier/ddpm-jax | config found: True


In [9]:
# Build params -> runs the key/shape + extra-modules asserts inside build_params
import yaml, jax
from src.models.model import DDPM
from src.train_ddpm import build_params

cfg = yaml.safe_load(open("configs/config.yaml"))
ddpm = DDPM(cfg)
params = build_params(ddpm.unet, cfg, jax.random.PRNGKey(0))
print("\u2705 build_params assertions passed")

/snap/google-cloud-cli/473/lib/third_party/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
Copying gs://ddpm-thesis-rh/checkpoints/ddpm/ckpt_epoch_0299.pkl to file:///tmp/tmpqqimv0fs.pkl
  
...

Average throughput: 212.7MiB/s


✅ build_params assertions passed


In [10]:
# Identity-init no-op: at init, conditioning must not change the output
import jax.numpy as jnp

model = ddpm.unet
img, in_ch = cfg["data"]["image_size"], cfg["model"]["in_ch"]
dum_x = jnp.ones((1, img, img, in_ch))
tt = jnp.ones((1,), dtype=jnp.int32)
out_none = model.apply({"params": params}, dum_x, tt, train=False, condRes=None)
out_cond = model.apply({"params": params}, dum_x, tt, train=False, condRes=jnp.ones_like(dum_x))
ok = bool(jnp.allclose(out_none, out_cond, atol=1e-5))
print("\u2705 identity-init no-op:" if ok else "\u274c conditioning changed output at init:", ok)

✅ identity-init no-op: True
